In [1]:
import logging
import time
from functools import wraps


logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s : %(message)s"
)

logger = logging.getLogger("api")


def timer(func):

    @wraps(func)
    def wrapper(*args, **kwargs):

        start = time.time()

        result = func(*args, **kwargs)

        end = time.time()
        duration = end - start

        logger.info(f"{func.__name__} executed in {duration:.4f} seconds")

        return result

    return wrapper

In [2]:
def log_io(func):

    @wraps(func)
    def wrapper(*args, **kwargs):

        logger.info(f"Calling {func.__name__}")
        logger.info(f"Arguments: {args} {kwargs}")

        result = func(*args, **kwargs)

        logger.info(f"Result: {result}")

        return result

    return wrapper


def require_role(role):

    def decorator(func):

        @wraps(func)
        def wrapper(*args, **kwargs):

            user = kwargs.get("user")

            if user["role"] != role:
                logger.warning("Access denied")
                return {"error": "Access denied"}

            logger.info("Access granted")

            return func(*args, **kwargs)

        return wrapper

    return decorator

In [3]:
@timer
@log_io
@require_role("admin")
def process_payment(amount, user):

    time.sleep(0.1)

    return {"status": "paid", "amount": amount}


@timer
@log_io
@require_role("member")
def get_profile(user):

    return {"name": user["name"], "role": user["role"]}


@timer
@log_io
def public_status():

    return {"status": "API running"}

In [4]:
def main():

    admin = {"name":"Haseeb","role":"admin"}
    member = {"name":"Arslan","role":"member"}
    guest = {"name":"Umair","role":"guest"}

    logger.info("Admin payment request")
    process_payment(amount=100, user=admin)

    logger.info("Guest payment request")
    process_payment(amount=50, user=guest)

    logger.info("Member profile request")
    get_profile(user=member)

    logger.info("Guest profile request")
    get_profile(user=guest)

    logger.info("Public API status")
    public_status()

In [5]:
main()